# UPI Fraud Detection - Preprocessing

This notebook prepares the dataset before model training.

Workflow:
1. Load the raw dataset
2. Drop identifier columns
3. Separate target and features
4. Handle missing values
5. Detect categorical columns
6. Encode categorical features

The output of this notebook will be a clean feature matrix ready for train/test split.

## 1. Load Libraries and Dataset

We start with the standard data science libraries and load the raw CSV file from the dataset folder.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

DATA_PATH = Path('..') / 'dataset' / 'upi_fraud_dataset_behavioral_v2.csv'
df = pd.read_csv(DATA_PATH)

df.head()

,transaction_id,transaction_date,transaction_time,customer_id,sender_upi,receiver_upi,amount,transaction_type,merchant_category,sender_account_age_days,receiver_account_age_days,device_id,device_changed,known_device,trusted_device,os_type,app_name,ip_address,location,location_changed,new_beneficiary,beneficiary_frequency,transactions_last_10_minutes,transactions_last_hour,transactions_last_24_hours,amount_sent_last_24_hours,amount_received_last_24_hours,unique_receivers_today,balance_before,balance_after,decimal_pattern,money_flow_pattern,customer_behaviour_deviation,is_fraud,fraud_type
0,TXN0096196,2026-04-07,00:07:47,CUST02924,ishaantank6535@okhdfcbank,vsingh6378@paytm,745.21,P2P,NaN,124,235,DEV-48EA6BD0FFEE,False,True,True,Android,Paytm,58.35.75.178,Pune,False,True,0,0,0,0,0.0,0.0,0,44705.71,43960.50,False,Refund,0.2219,0,NaN
1,TXN0067907,2026-04-07,00:08:45,CUST04012,harinakshikar7902@oksbi,bahadurjitchandran9738@okhdfcbank,1230.44,P2M,Electronics,308,146,DEV-B7CD99AC4D9B,False,True,True,Android,Paytm,171.220.144.175,Mumbai,False,True,0,0,0,0,0.0,0.0,0,1269254.65,1268024.21,False,Travel Payment,0.1813,0,NaN
2,TXN0008569,2026-04-07,00:10:50,CUST02392,sathvikswamy9935@paytm,lavanyasen2814@okaxis,643.95,P2P,NaN,113,322,DEV-CDC3A225D3F6,False,True,True,Android,Paytm,130.196.162.207,Lucknow,False,True,0,0,0,0,0.0,0.0,0,343755.85,343111.90,False,Gift,0.4219,0,NaN
3,TXN0083623,2026-04-07,00:13:39,CUST01883,aayushwali4880@paytm,finnchanda3055@okicici,203.08,P2P,NaN,627,1098,DEV-3766EBCA617C,False,True,True,iOS,AmazonPay,79.123.13.235,Nagpur,False,True,0,0,0,0,0.0,0.0,0,261606.72,261403.64,False,Personal Transfer,0.2455,0,NaN
4,TXN0060281,2026-04-07,00:20:18,CUST04883,amritalalla5990@okicici,merchant8860@apl,401.43,P2M,Rent,1000,266,DEV-B50D1ADAFF11,False,True,True,Android,BHIM,92.106.65.184,Jaipur,False,True,0,0,0,0,0.0,0.0,0,120217.53,119816.10,False,Bills Payment,0.1768,0,NaN


## 2. Drop Identifier Columns

These columns identify a transaction or account, but they do not directly describe fraud behavior. Keeping them can make the model memorize records instead of learning patterns.

In [2]:
target_column = 'is_fraud'
identifier_columns = [
    'transaction_id',
    'customer_id',
    'sender_upi',
    'receiver_upi',
    'device_id',
    'ip_address',
    'fraud_type',
]

available_identifier_columns = [column for column in identifier_columns if column in df.columns]
processed_df = df.drop(columns=available_identifier_columns).copy()

print('Dropped columns:')
print(available_identifier_columns)
print('Remaining columns:')
print(processed_df.columns.tolist())

Dropped columns:
['transaction_id', 'customer_id', 'sender_upi', 'receiver_upi', 'device_id', 'ip_address', 'fraud_type']
Remaining columns:
['transaction_date', 'transaction_time', 'amount', 'transaction_type', 'merchant_category', 'sender_account_age_days', 'receiver_account_age_days', 'device_changed', 'known_device', 'trusted_device', 'os_type', 'app_name', 'location', 'location_changed', 'new_beneficiary', 'beneficiary_frequency', 'transactions_last_10_minutes', 'transactions_last_hour', 'transactions_last_24_hours', 'amount_sent_last_24_hours', 'amount_received_last_24_hours', 'unique_receivers_today', 'balance_before', 'balance_after', 'decimal_pattern', 'money_flow_pattern', 'customer_behaviour_deviation', 'is_fraud']


## 3. Separate Features and Target

`is_fraud` is the label we want to predict. Everything else becomes part of the feature matrix.

In [ ]:
y = processed_df[target_column].copy()
X = processed_df.drop(columns=[target_column]).copy()

transaction_date = pd.to_datetime(X['transaction_date'], errors='coerce')
X['transaction_dayofweek'] = transaction_date.dt.dayofweek
X['transaction_day'] = transaction_date.dt.day
X['transaction_month'] = transaction_date.dt.month
X['transaction_year'] = transaction_date.dt.year
X['transaction_hour'] = pd.to_datetime(X['transaction_time'], format='%H:%M:%S', errors='coerce').dt.hour
X = X.drop(columns=['transaction_date', 'transaction_time'])

print(f'Feature matrix shape: {X.shape}')
print(f'Target shape: {y.shape}')
print('Target distribution:')
print(y.value_counts().rename(index={0: 'Legit', 1: 'Fraud'}))

Feature matrix shape: (100000, 27)
Target shape: (100000,)
Target distribution:
is_fraud
Legit    95000
Fraud     5000
Name: count, dtype: int64


## 4. Handle Missing Values

For categorical fields like `merchant_category`, we can fill missing values with a clear placeholder such as `P2P` or `NONE`. For numeric columns, we use the median because it is more stable than the mean when outliers exist.

In [4]:
categorical_columns = X.select_dtypes(include=['object', 'bool']).columns.tolist()
numeric_columns = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

for column in categorical_columns:
    if column == 'merchant_category':
        X[column] = X[column].fillna('P2P')
    else:
        X[column] = X[column].fillna('NONE')

for column in numeric_columns:
    X[column] = X[column].fillna(X[column].median())

print('Missing values after cleanup:')
print(X.isna().sum()[X.isna().sum() > 0] if (X.isna().sum() > 0).any() else 'No missing values left.')

Missing values after cleanup:
No missing values left.


## 5. Detect Categorical Features

These columns must be encoded into numeric values before a machine learning model can train on them.

In [5]:
categorical_columns = X.select_dtypes(include=['object', 'bool']).columns.tolist()
print('Categorical columns:')
print(categorical_columns)

numeric_columns = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
print('Numeric columns:')
print(numeric_columns)

Categorical columns:
['transaction_date', 'transaction_time', 'transaction_type', 'merchant_category', 'device_changed', 'known_device', 'trusted_device', 'os_type', 'app_name', 'location', 'location_changed', 'new_beneficiary', 'decimal_pattern', 'money_flow_pattern']
Numeric columns:
['amount', 'sender_account_age_days', 'receiver_account_age_days', 'beneficiary_frequency', 'transactions_last_10_minutes', 'transactions_last_hour', 'transactions_last_24_hours', 'amount_sent_last_24_hours', 'amount_received_last_24_hours', 'unique_receivers_today', 'balance_before', 'balance_after', 'customer_behaviour_deviation']


## 6. Encode Categorical Columns

We use one-hot encoding so each category becomes a separate binary feature. This works well for tree-based models and linear models when the number of categories is manageable.

In [6]:
X_encoded = pd.get_dummies(X, columns=categorical_columns, drop_first=False)

print(f'Original feature shape: {X.shape}')
print(f'Encoded feature shape: {X_encoded.shape}')
print('Preview of encoded columns:')
print(X_encoded.columns[:25].tolist())

Original feature shape: (100000, 27)
Encoded feature shape: (100000, 52037)
Preview of encoded columns:
['amount', 'sender_account_age_days', 'receiver_account_age_days', 'beneficiary_frequency', 'transactions_last_10_minutes', 'transactions_last_hour', 'transactions_last_24_hours', 'amount_sent_last_24_hours', 'amount_received_last_24_hours', 'unique_receivers_today', 'balance_before', 'balance_after', 'customer_behaviour_deviation', 'transaction_date_2026-04-07', 'transaction_date_2026-04-08', 'transaction_date_2026-04-09', 'transaction_date_2026-04-10', 'transaction_date_2026-04-11', 'transaction_date_2026-04-12', 'transaction_date_2026-04-13', 'transaction_date_2026-04-14', 'transaction_date_2026-04-15', 'transaction_date_2026-04-16', 'transaction_date_2026-04-17', 'transaction_date_2026-04-18']


## 7. Ready for Train/Test Split

At this point, the data is cleaned and encoded. The next notebook can split the data into training and testing sets, then train and compare models.